# [8.5] Sparse Feature Circuits

This notebook is the learner surface for the sparse-feature circuit validation ladder. The page gives the longer explanation; this notebook keeps the runnable implementation cells and visible tests together.

<details>
<summary>Expected output</summary>

As you complete each implementation cell, the corresponding visible test should print `All tests in ... passed!`.

</details>

<details>
<summary>Help - how to use this notebook</summary>

Run the setup cells once, implement one exercise at a time, and run the visible tests immediately after each implementation. Do not edit the report-backed final metrics; regenerate `verification_report.json` with the script when validating CUDA evidence.

</details>


In [ ]:
import json
import math
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter8_automated_circuits"
section = "part5_sparse_feature_circuits"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part5_sparse_feature_circuits.tests as tests
import part5_sparse_feature_circuits.utils as utils

EXERCISE_ID = "8_5_sparse_feature_circuits"
GT_TIER = "GT-0"
DIFFICULTY = 4
IMPORTANCE = 5
EXPECTED_RUNTIME = "seconds on toy contract; minutes on local real-model path"
REQUIRES_GPU = True


In [ ]:
@dataclass(frozen=True)
class FeatureNodePatchingReport:
    selected_feature_ids: tuple[int, ...]
    full_logit_diff: float
    graph_logit_diff: float
    recovered_fraction: float
    passes_recovery: bool


@dataclass(frozen=True)
class FeatureEdgePatchingReport:
    selected_edges: tuple[tuple[int, int], ...]
    full_edge_score: float
    graph_edge_score: float
    recovered_fraction: float
    passes_recovery: bool


@dataclass(frozen=True)
class EAPIGComparisonReport:
    exact_error: float
    eap_error: float
    eap_ig_error: float
    eap_passes: bool
    eap_ig_improves: bool


@dataclass(frozen=True)
class FeatureGraphThresholdReport:
    selected_feature_ids: tuple[int, ...]
    threshold: float
    full_logit_diff: float
    graph_logit_diff: float
    recovered_fraction: float
    passes_threshold: bool


@dataclass(frozen=True)
class RandomFeatureGraphControlReport:
    target_graph_logit_diff: float
    random_graph_logit_diff: float
    target_recovered_fraction: float
    random_recovered_fraction: float
    margin: float
    random_graph_fails: bool


@dataclass(frozen=True)
class SparseFeatureEditingReport:
    target_feature_ids: tuple[int, ...]
    spurious_feature_ids: tuple[int, ...]
    random_feature_ids: tuple[int, ...]
    baseline_train_accuracy: float
    baseline_ood_accuracy: float
    edited_train_accuracy: float
    edited_ood_accuracy: float
    random_edit_ood_accuracy: float
    black_box_baseline_ood_accuracy: float
    spurious_reliance_before: float
    spurious_reliance_after: float
    target_reliance_after: float
    ood_improvement: float
    random_edit_improvement: float
    target_accuracy_drop: float
    spurious_reliance_decreases: bool
    target_task_preserved: bool
    ood_generalization_improves: bool
    random_edit_control_fails: bool
    editing_passes: bool


In [ ]:
def _index_tensor(indices: t.Tensor | list[int] | tuple[int, ...], *, device: t.device) -> t.Tensor:
    if isinstance(indices, t.Tensor):
        return indices.to(device=device, dtype=t.long).flatten()
    return t.tensor(list(indices), device=device, dtype=t.long)


def _require_finite_tensor(name: str, tensor: t.Tensor) -> t.Tensor:
    if tensor.numel() == 0:
        raise ValueError(f"{name} must be non-empty.")
    if not t.isfinite(tensor.float()).all():
        raise ValueError(f"{name} must contain only finite values.")
    return tensor


def _require_finite_nonnegative(name: str, value: float) -> float:
    value_float = float(value)
    if not math.isfinite(value_float):
        raise ValueError(f"{name} must be finite.")
    if value_float < 0:
        raise ValueError(f"{name} must be non-negative.")
    return value_float


def _validate_index_tensor(ids: t.Tensor, *, name: str, upper_bound: int) -> t.Tensor:
    if ids.numel() == 0:
        raise ValueError(f"at least one {name} id is required.")
    if ids.min().item() < 0 or ids.max().item() >= upper_bound:
        raise ValueError(f"{name} id is out of range.")
    if ids.unique().numel() != ids.numel():
        raise ValueError(f"{name} ids must be unique.")
    return ids


def _fraction(numerator: float, denominator: float) -> float:
    if denominator == 0:
        raise ValueError("reference score must be nonzero.")
    return numerator / denominator


def toy_sparse_feature_fixture(device: str | t.device = "cpu") -> dict[str, t.Tensor]:
    device = t.device(device)
    return {
        "feature_contributions": t.tensor([0.7, 0.2, 0.05, 0.05], device=device),
        "edge_scores": t.tensor([[0.05, 0.8], [0.05, 0.1]], device=device),
        "exact_scores": t.tensor([0.7, 0.2, 0.05, 0.05], device=device),
        "eap_scores": t.tensor([0.5, 0.35, 0.1, 0.05], device=device),
        "eap_ig_scores": t.tensor([0.69, 0.21, 0.04, 0.06], device=device),
    }


## Validation Loop

The section builds from exact toy oracles to report-backed CUDA evidence:

1. Exact sparse-feature node and edge patching.
2. EAP/EAP-IG approximation checks against exact scores.
3. Thresholded feature graphs and same-size random graph controls.
4. SHIFT-style sparse-feature editing on generated data.
5. Official artifact and held-out faithfulness checks in `verification_report.json`.

<details>
<summary>Common bugs</summary>

- Letting random controls overlap the target features.
- Passing NaN/Inf scores through a report and only checking final booleans.
- Treating artifact readiness as paper-level sparse-feature circuit replication.

</details>


## Exercise 1 - Exact node patching

Implement a feature-node oracle that reports how much of the full logit-difference effect is recovered by selected feature IDs.

<details>
<summary>Expected output</summary>

```text
All tests in `test_exact_feature_node_patching_report_recovers_selected_features` passed!
```

</details>

<details>
<summary>Help - denominator</summary>

The denominator is the sum of all feature contributions, not the number of selected features.

</details>

<details>
<summary>Solution</summary>

See the solution notebook after attempting the implementation.

</details>

In [ ]:
def exact_feature_node_patching_report(
    feature_contributions: t.Tensor,
    feature_ids: t.Tensor | list[int] | tuple[int, ...],
    *,
    min_recovered_fraction: float = 0.75,
) -> FeatureNodePatchingReport:
    raise NotImplementedError()


tests.test_exact_feature_node_patching_report_recovers_selected_features(
    exact_feature_node_patching_report,
)

## Exercise 2 - Exact edge patching

Implement the source-feature edge oracle using absolute edge magnitudes.

<details>
<summary>Expected output</summary>

```text
All tests in `test_exact_feature_edge_patching_report_recovers_selected_edges` passed!
```

</details>

<details>
<summary>Help - edge IDs</summary>

Each selected edge is a `(source_id, feature_id)` pair. Validate both axes and reject duplicates.

</details>

<details>
<summary>Solution</summary>

See the solution notebook after attempting the implementation.

</details>

In [ ]:
def exact_feature_edge_patching_report(
    edge_scores: t.Tensor,
    selected_edges: list[tuple[int, int]] | tuple[tuple[int, int], ...],
    *,
    min_recovered_fraction: float = 0.75,
) -> FeatureEdgePatchingReport:
    raise NotImplementedError()


tests.test_exact_feature_edge_patching_report_recovers_selected_edges(
    exact_feature_edge_patching_report,
)

## Exercise 3 - EAP versus EAP-IG

Compare approximation error against exact patching scores.

<details>
<summary>Expected output</summary>

```text
All tests in `test_eap_ig_comparison_report_improves_over_plain_eap` passed!
```

</details>

<details>
<summary>Help - improvement gate</summary>

EAP-IG must have lower mean absolute error than EAP and stay under the configured error threshold.

</details>

<details>
<summary>Solution</summary>

See the solution notebook after attempting the implementation.

</details>

In [ ]:
def eap_ig_comparison_report(
    exact_scores: t.Tensor,
    eap_scores: t.Tensor,
    eap_ig_scores: t.Tensor,
    *,
    max_eap_ig_error: float = 0.05,
) -> EAPIGComparisonReport:
    raise NotImplementedError()


tests.test_eap_ig_comparison_report_improves_over_plain_eap(eap_ig_comparison_report)

## Exercise 4 - Thresholded graph and random control

Build a thresholded graph, then compare it to a disjoint same-size random graph.

<details>
<summary>Expected output</summary>

```text
All tests in `test_threshold_feature_graph_report_keeps_large_features` passed!
All tests in `test_random_feature_graph_control_report_rejects_random_graph` passed!
```

</details>

<details>
<summary>Help - controls</summary>

The random graph must be the same size as the target graph and must not overlap it.

</details>

<details>
<summary>Solution</summary>

See the solution notebook after attempting the implementation.

</details>

In [ ]:
def threshold_feature_graph_report(
    feature_contributions: t.Tensor,
    *,
    threshold: float,
    min_recovered_fraction: float = 0.8,
) -> FeatureGraphThresholdReport:
    raise NotImplementedError()


def random_feature_graph_control_report(
    feature_contributions: t.Tensor,
    target_feature_ids: t.Tensor | list[int] | tuple[int, ...],
    random_feature_ids: t.Tensor | list[int] | tuple[int, ...],
    *,
    min_margin: float = 0.2,
) -> RandomFeatureGraphControlReport:
    raise NotImplementedError()


tests.test_threshold_feature_graph_report_keeps_large_features(threshold_feature_graph_report)
tests.test_random_feature_graph_control_report_rejects_random_graph(
    random_feature_graph_control_report,
)

## Exercise 5 - SHIFT-style sparse-feature editing

Suppress a spurious feature while preserving target accuracy and improving OOD accuracy.

<details>
<summary>Expected output</summary>

```text
All tests in `test_shift_style_sparse_feature_editing_report_removes_spurious_feature` passed!
```

</details>

<details>
<summary>Help - what this does not claim</summary>

This is generated sparse-feature data, not real-model debiasing. The exercise teaches the intervention safety contract.

</details>

<details>
<summary>Solution</summary>

See the solution notebook after attempting the implementation.

</details>

In [ ]:
def shift_style_sparse_feature_editing_report(
    train_features: t.Tensor,
    train_labels: t.Tensor,
    ood_features: t.Tensor,
    ood_labels: t.Tensor,
    classifier_weights: t.Tensor,
    *,
    target_feature_ids: t.Tensor | list[int] | tuple[int, ...],
    spurious_feature_ids: t.Tensor | list[int] | tuple[int, ...],
    random_feature_ids: t.Tensor | list[int] | tuple[int, ...],
    suppression: float = 0.0,
    max_target_accuracy_drop: float = 0.05,
    min_ood_improvement: float = 0.5,
    min_random_edit_gap: float = 0.5,
) -> SparseFeatureEditingReport:
    raise NotImplementedError()


tests.test_shift_style_sparse_feature_editing_report_removes_spurious_feature(
    shift_style_sparse_feature_editing_report,
)

## Signature Result

The accepted local report gives the following signature result:

| Check | Result |
|---|---:|
| Toy node recovery | 0.9000 |
| Toy edge recovery | 0.8000 |
| EAP-IG error | 0.0100 |
| Pythia residual recovery vs random | 0.3759 vs 0.0301 |
| Official SAE attribution recovery vs random | 0.5829 vs 0.0000 |
| Official graph artifact | 37 nodes, 59 edges |
| Held-out faithfulness | 1.0000 on 40 `simple_test` examples |
| SHIFT-style edit | OOD 0.0 to 1.0, train drop 0.0 |

<details>
<summary>Interpreting the result</summary>

The toy ladder validates your implementation contract. The CUDA report validates the local model/artifact path. The report does not claim a full paper-level sparse-feature circuit replication.

</details>

## Limitations

- The exercise implementations are toy and generated-data contracts.
- The residual-feature preflight is not an official SAE graph replication.
- The SHIFT-style edit is not a real-model debiasing result.

## Further Research

- Build a fully student-executed SAE graph on a small prompt batch.
- Compare graph faithfulness across thresholds and random-control samplers.


In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    metrics = report["metrics"]["gpu_test"]
    assert report["accepted"]
    assert report["peak_vram_gb"] <= max_vram_gb
    return metrics


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)
